# TSV counts → BigWig
Convert raw count TSV (chrom/start/end/ES_count/MS_count/LS_count/G1_count) to BigWig tracks
using the same TPM normalization as the training pipeline, so values are on the same scale
as model predictions.

In [ ]:
import numpy as np
import pandas as pd
import pyBigWig
from pathlib import Path

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
TSV_PATH   = '/Users/lzz/Downloads/rice_wt_counts_1024bp.tsv'
OUTPUT_DIR = 'output/ground_truth'

In [ ]:
df = pd.read_csv(TSV_PATH, sep='\t')
if '#chrom' in df.columns:
    df = df.rename(columns={'#chrom': 'chrom'})

# Global TPM normalization — same as load_labels in data_utils.py
def tpm_normalize(counts):
    total = counts.sum()
    if total == 0:
        return np.zeros_like(counts, dtype=np.float32)
    return (counts / total * 1e6).astype(np.float32)

es = tpm_normalize(df['ES_count'].values)
ms = tpm_normalize(df['MS_count'].values)
ls = tpm_normalize(df['LS_count'].values)
g1 = tpm_normalize(df['G1_count'].values)

# WRT — same formula as compute_wrt in data_utils.py
eps = 1e-6
wrt = (0.5 * ms + ls) / (es + ms + ls + eps)

df['ES_tpm'], df['MS_tpm'], df['LS_tpm'], df['G1_tpm'] = es, ms, ls, g1
df['WRT'] = wrt

print(f'Loaded {len(df):,} bins from {TSV_PATH}')
print(df[['chrom','start','end','ES_tpm','MS_tpm','LS_tpm','G1_tpm','WRT']].head())

In [ ]:
# Chromosome sizes inferred from TSV (max end per chrom)
chrom_sizes = (
    df.groupby('chrom')['end'].max()
    .sort_index()
    .to_dict()
)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

TRACKS = {
    'ES':  'ES_tpm',
    'MS':  'MS_tpm',
    'LS':  'LS_tpm',
    'G1':  'G1_tpm',
    'WRT': 'WRT',
}

for track, col in TRACKS.items():
    bw = pyBigWig.open(str(out_dir / f'{track}.bw'), 'w')
    bw.addHeader(list(chrom_sizes.items()))
    for chrom, grp in df.sort_values(['chrom', 'start']).groupby('chrom', sort=False):
        chroms = [chrom] * len(grp)
        starts = grp['start'].tolist()
        ends   = grp['end'].tolist()
        values = grp[col].astype(float).tolist()
        bw.addEntries(chroms, starts, ends=ends, values=values)
    bw.close()
    print(f'Written {track}.bw')

print(f'Done — {len(TRACKS)} BigWig tracks in {OUTPUT_DIR}')